In [ ]:
# ==============================================================================
# LIBRARY OF CONGRESS SUBJECT HEADINGS (LCSH) - UNIFIED INGESTION
# 
# ARCHITECTURE OVERVIEW:
# This script performs a targeted downward crawl of LCSH starting from both 
# broad conceptual seeds (e.g., Religion) and specific taxonomic seeds (e.g., Sects).
#
# INGESTION STRATEGY:
# 1. Shortest-Path BFS: Explores parents level-by-level to guarantee the 
#    shortest path back to a core religious seed, preventing semantic drift.
# 2. Dynamic Absolute Pathing: Once a religious seed is reached, the script 
#    dynamically climbs the API to the absolute LCSH top (e.g. Auxiliary sciences 
#    of history) to ensure consistent, complete Hierarchy_Paths.
# 3. Structural Guardrails: Aggressively rejects 'ComplexSubject' nodes and 
#    compound strings (--) to maintain conceptual purity.
# 4. Translucent Crawling: Bypasses metadata extraction for existing nodes 
#    while continuing downward recursion to capture deeper descendants.
# ==============================================================================

import os
import sys
import requests
import pandas as pd
import time
from collections import deque
from dotenv import load_dotenv

# --- 0. Global State Reset ---
path_cache = {}
api_cache = {} 
ancestry_cache = {} 
retry_queue = [] 

# --- 1. Environment & Directory Setup ---
load_dotenv("../../config/.env")
contact_email = os.getenv("CONTACT_EMAIL", "anonymous@example.com")

notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
interim_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "interim"))
error_log_dir = os.path.abspath(os.path.join(interim_data_dir, "logs", "errors"))
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))

os.makedirs(raw_data_dir, exist_ok=True)
os.makedirs(error_log_dir, exist_ok=True)

sys.path.append(config_dir)
from ingest_schema_manager import get_registry_info, finalize_row, COLUMN_ORDER

# --- 2. Source & Registry Configuration ---
SOURCE_PREFIX = "LCSH"
registry_data = get_registry_info(prefix=SOURCE_PREFIX, config_dir=config_dir)
SOURCE_NAME = registry_data['Source_Name']
BASE_URI = registry_data['Base_URI']

raw_ingest_file = os.path.join(raw_data_dir, "raw_lcsh.csv")
drift_report_file = os.path.join(interim_data_dir, "lcsh_drift_candidates.csv")
boundary_report_file = os.path.join(interim_data_dir, "lcsh_boundary_candidates.csv")
hard_failure_file = os.path.join(error_log_dir, "lcsh_hard_failures.csv")

HEADERS = {'User-Agent': f'ReligiousMappingProject/1.0 (mailto:{contact_email})', 'Accept': 'application/json'}

# --- 3. Seeds, Roots, and Guardrails ---
SEED_URIS = [
    "http://id.loc.gov/authorities/subjects/sh85112599",  # Religions
    "http://id.loc.gov/authorities/subjects/sh85112549",  # Religion
    "http://id.loc.gov/authorities/subjects/sh2005004944", # Religious adherents
    "http://id.loc.gov/authorities/subjects/sh85119451",  # Sects
    "http://id.loc.gov/authorities/subjects/sh85034719",  # Cults
    "http://id.loc.gov/authorities/subjects/sh85068280",  # Irreligion
    "http://id.loc.gov/authorities/subjects/sh85009109",  # Atheism
    "http://id.loc.gov/authorities/subjects/sh85085279",  # Monasticism and religious orders
    "http://id.loc.gov/authorities/subjects/sh85106171"   # Preaching
]

CORE_ROOT_LABELS = ["Religion", "Religions", "Religious adherents", "Sects", "Cults", 
                    "Irreligion", "Atheism", "Monasticism and religious orders", "Preaching"]

KNOWN_DRIFT_URIS = {"http://id.loc.gov/authorities/subjects/sh85108459"} # Psychology
KNOWN_TERMINAL_URIS = {"http://id.loc.gov/authorities/subjects/sh85081416"} # Marriage

MAX_DEPTH = 7
global_do_not_crawl = set()

# --- 4. Persistence ---
if os.path.exists(raw_ingest_file):
    existing_df = pd.read_csv(raw_ingest_file, dtype={'Concept_ID': str})
    if list(existing_df.columns) == COLUMN_ORDER:
        global_do_not_crawl.update(existing_df['URI'].dropna().tolist())

# --- 5. Helper Functions ---

def fetch_json(url):
    if url in api_cache: return api_cache[url]
    time.sleep(0.6) 
    for attempt in range(1, 4):
        try:
            res = requests.get(url, headers=HEADERS, timeout=30)
            if res.status_code == 200:
                data = res.json()
                api_cache[url] = data
                return data
            if res.status_code == 404: return None
        except: pass
        time.sleep(2 ** attempt)
    return None

def get_absolute_ancestry(uri, depth=0):
    """Recursive upward climb to the absolute top of the LCSH tree."""
    clean_uri = uri.rstrip('/').replace('.json', '')
    if clean_uri in ancestry_cache: return ancestry_cache[clean_uri]
    if depth > 15: return clean_uri.split('/')[-1]
        
    data = fetch_json(f"{clean_uri}.json")
    if not data: return ""
    
    node = next((item for item in data if isinstance(item, dict) and item.get('@id') == clean_uri), {})
    label = clean_uri.split('/')[-1]
    for pref in node.get('http://www.w3.org/2004/02/skos/core#prefLabel', []):
        if pref.get('@language', 'en').startswith('en'): label = pref.get('@value', label)

    broader = node.get('http://www.w3.org/2004/02/skos/core#broader', []) or \
              node.get('http://www.loc.gov/mads/rdf/v1#hasBroaderAuthority', [])
    
    if broader and broader[0].get('@id'):
        parent_path = get_absolute_ancestry(broader[0].get('@id'), depth + 1)
        full_path = f"{parent_path} > {label}" if parent_path else label
        ancestry_cache[clean_uri] = full_path
        return full_path
    return label

def get_best_religious_path(start_uri):
    """BFS for the shortest path to a religious root, then climbs to absolute root."""
    clean_start_uri = start_uri.rstrip('/').replace('.json', '')
    if clean_start_uri in path_cache: return path_cache[clean_start_uri]
    
    queue = deque([(clean_start_uri, [], 0)])
    visited = {clean_start_uri}
    
    while queue:
        curr_uri, path_labels, d = queue.popleft()
        if d > 12: continue
            
        data = fetch_json(f"{curr_uri}.json")
        if not data: continue
        
        node = next((item for item in data if isinstance(item, dict) and item.get('@id') == curr_uri), {})
        label = curr_uri.split('/')[-1]
        for pref in node.get('http://www.w3.org/2004/02/skos/core#prefLabel', []):
            if pref.get('@language', 'en').startswith('en'): label = pref.get('@value', label)

        # Disqualify branches that route through complex pre-coordinated strings
        is_complex = "http://www.loc.gov/mads/rdf/v1#ComplexSubject" in node.get('@type', []) or "--" in label
        if is_complex and d > 0: continue

        if label in CORE_ROOT_LABELS:
            absolute_top = get_absolute_ancestry(curr_uri)
            final_path = f"{absolute_top} > {' > '.join(path_labels[::-1])}" if path_labels else absolute_top
            path_cache[clean_start_uri] = final_path
            return final_path

        broader = node.get('http://www.w3.org/2004/02/skos/core#broader', []) or \
                  node.get('http://www.loc.gov/mads/rdf/v1#hasBroaderAuthority', [])
        for parent in broader:
            p_uri = parent.get('@id')
            if p_uri and p_uri not in visited:
                visited.add(p_uri)
                new_labels = list(path_labels); new_labels.append(label)
                queue.append((p_uri, new_labels, d + 1))
    return None

# --- 6. Core Extraction Logic ---

def process_lc_node(uri, p_id="", current_depth=0, is_retry_pass=False):
    clean_uri = uri.rstrip('/').replace('.json', '')
    if clean_uri in KNOWN_DRIFT_URIS: return
    
    is_already_ingested = clean_uri in global_do_not_crawl
    if is_already_ingested and current_depth >= MAX_DEPTH: return
    
    data = fetch_json(f"{clean_uri}.json")
    if not data:
        if not is_retry_pass: retry_queue.append({'uri': clean_uri, 'p_id': p_id, 'depth': current_depth})
        return

    node_data = next((item for item in data if isinstance(item, dict) and item.get('@id') == clean_uri), {})
    if not node_data: return

    label = "No Label"
    for pref in node_data.get('http://www.w3.org/2004/02/skos/core#prefLabel', []):
        if pref.get('@language', 'en').startswith('en'): label = pref.get('@value', label)

    # GATE 1: Prune Complex Subjects
    if "http://www.loc.gov/mads/rdf/v1#ComplexSubject" in node_data.get('@type', []) or "--" in label:
        return

    # GATE 2: Ancestry Guardrail
    valid_path = get_best_religious_path(clean_uri)
    if not valid_path and current_depth > 0:
        return

    cid = clean_uri.split('/')[-1]
    if not is_already_ingested:
        print(f"\r[DEPTH {current_depth}] Ingesting: {label[:50]:<50}", end="", flush=True)
        
        extracted = {
            'Source_System': SOURCE_NAME, 'Primary_Label': label, 'CURIE': f"LCSH:{cid}",
            'Formal_Label': label, 'Concept_Type': 'skos:Concept', 'Hierarchy_Path': valid_path,
            'Parent_IDs': str(p_id), 'Concept_ID': str(cid), 'URI': clean_uri, 'Status': 'active'
        }
        
        pd.DataFrame([finalize_row(extracted)])[COLUMN_ORDER].to_csv(
            raw_ingest_file, mode='a', index=False, header=not os.path.isfile(raw_ingest_file), encoding='utf-8-sig'
        )
        global_do_not_crawl.add(clean_uri)
    else:
        print(f"\r[DEPTH {current_depth}] Passing Through: {label[:45]:<45}", end="", flush=True)

    if clean_uri in KNOWN_TERMINAL_URIS: return

    narrower = node_data.get('http://www.w3.org/2004/02/skos/core#narrower', [])
    if narrower and current_depth < MAX_DEPTH:
        for child in narrower:
            c_uri = child.get('@id')
            if c_uri and c_uri.startswith(BASE_URI):
                process_lc_node(c_uri, p_id=cid, current_depth=current_depth + 1, is_retry_pass=is_retry_pass)

# --- 7. Execution ---
print(f"Starting Unified BFS Ingestion (Target Depth: {MAX_DEPTH})...\n")
for seed_uri in SEED_URIS:
    process_lc_node(seed_uri, current_depth=0)

if retry_queue:
    print(f"\n\nPhase 2 Cleanup...")
    for task in list(retry_queue):
        process_lc_node(task['uri'], p_id=task['p_id'], current_depth=task['depth'], is_retry_pass=True)

print(f"\n\nDone. Final data in {raw_ingest_file}")